# WaveTrader — Evaluation and Backtesting from Checkpoint

This notebook simulates a "production ready" environment where you mount your Google Drive, download your trained model checkpoint, pull the latest `wavetrader` code from GitHub, and run a full evaluation on the hold-out test set.

---
**Runtime Required:** CPU or GPU (GPU recommended for faster batch evaluation).

**Pre-requisites:**
1. You must have already trained a model using the main training notebook.
2. The checkpoint must exist in `MyDrive/phase_lm/checkpoints/`.
3. Preprocessed parquet data must exist in `MyDrive/phase_lm/processed_data/test/`.

## 1. Setup Workspace and Mount Google Drive

In [ ]:
# ── Workspace Setup ───────────────────────────────────────────────────────────
from google.colab import drive
import os, pathlib

drive.mount("/content/drive")

DRIVE_ROOT   = pathlib.Path("/content/drive/MyDrive/phase_lm")
DRIVE_CKPT   = DRIVE_ROOT / "checkpoints"   
DRIVE_PROC   = DRIVE_ROOT / "processed_data" 

LOCAL_ROOT = pathlib.Path("/content/phase_lm")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted.")
print(f"Checkpoints dir  : {DRIVE_CKPT}")
print(f"Processed subset : {DRIVE_PROC / 'test'}")

## 2. Clone Repository and Pull Latest Changes
We'll clone the repository if it doesn't exist locally, pull the latest changes, and install the required dependencies.

In [ ]:
# ── Clone Repository ────────────────────────────────────────────────────────────
import os
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/phase_lm.git" # IMPORTANT: Update with your repo URL
REPO_DIR = "/content/phase_lm"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository exists. Pulling latest changes...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "stash"], check=False)
    subprocess.run(["git", "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
sys.path.append(REPO_DIR)

print("Installing dependencies...")
!pip install -q ta pandas_ta scikit-learn
!pip install -q -r requirements.txt || echo "No requirements.txt found or installed dependencies successfully."
print("Setup Complete.")

## 3. Load Model Configuration and Weights
Load the serialized JSON config created during training, reconstruct the model architecture dynamically (handling both `MTFConfig` and standard configs), and inject the trained `.pt` saved state.

In [ ]:
# ── Load Model Config and Weights ─────────────────────────────────────────────
import torch
import json
from wavetrader.model import WaveTraderMTF, FluxSignal
from wavetrader.config import MTFConfig, SignalConfig

CKPT_DIR = DRIVE_CKPT / "best_model"
CONFIG_PATH = CKPT_DIR / "config.json"
WEIGHTS_PATH = CKPT_DIR / "model_weights.pt"

print(f"Loading config from {CONFIG_PATH}")
with open(CONFIG_PATH, "r") as f:
    cfg_dict = json.load(f)

# Reconstruct Model and Config Logic
is_mtf = "timeframes" in cfg_dict
config = MTFConfig(**cfg_dict) if is_mtf else SignalConfig(**cfg_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

if is_mtf:
    model = WaveTraderMTF(config).to(device)
    print("Initialized MTF WaveTrader model.")
else:
    model = FluxSignal(config).to(device)
    print("Initialized single timeframe FluxSignal model.")

print(f"Loading weights from {WEIGHTS_PATH}")
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
model.eval()
print("Model dynamically restored and evaluation ready.")

## 4. Load Test Data
Load and prepare the unseen testing batch using the parsed `config` layout.

In [ ]:
# ── Prepare Data ──────────────────────────────────────────────────────────────
import torch.utils.data
from wavetrader.data import MTFForexDataset, ForexDataset

print(f"Loading test data from {DRIVE_PROC / 'test'}...")

if is_mtf:
    test_dataset = MTFForexDataset(DRIVE_PROC / "test", config)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
    print(f"Loaded Multi-Timeframe dataloader with timeframes: {config.timeframes}")
else:
    test_dataset = ForexDataset(DRIVE_PROC / "test", config, mode="test")
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
    print(f"Loaded Single-Timeframe dataloader on timeframe: {config.timeframe}")


## 5. Validation Tests
Execute a validation pass against the newly structured validation/holdout sets explicitly extracting loss and F1.

In [ ]:
# ── Execute Validation ────────────────────────────────────────────────────────
from wavetrader.training import eval_epoch
import torch.nn as nn

print("Evaluating test dataset natively...")

try:
    criterion = nn.CrossEntropyLoss()
    val_loss, val_acc, val_f1 = eval_epoch(model, test_loader, criterion, device, is_mtf=is_mtf)

    print("-" * 40)
    print(f"Holdout Validation Results:")
    print(f" Loss     : {val_loss:.4f}")
    print(f" Accuracy : {val_acc:.4f} ({val_acc * 100:.2f}%)")
    print(f" F1 Score : {val_f1:.4f}")
    print("-" * 40)
except Exception as e:
    print(f"Validation Evaluation Error: {e}")

In [ ]:
# ── Detailed Visual Logs & Confusion Matrix ───────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

model.eval()
all_preds, all_labels, all_confs = [], [], []

print("Running deep inference for visual telemetry...")
with torch.no_grad():
    for batch in test_loader:
        if not is_mtf:
            batch_gpu = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        else:
            batch_gpu = {k: {feat: v.to(device) for feat, v in val.items() if isinstance(v, torch.Tensor)} 
                         if isinstance(val, dict) else val.to(device) if isinstance(val, torch.Tensor) else val
                         for k, val in batch.items()}
        
        labels = batch_gpu.pop("label")
        out    = model(batch_gpu)
        preds  = out["signal_logits"].argmax(-1)
        confs  = out["confidence"].squeeze(-1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_confs.append(confs.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_confs  = torch.cat(all_confs).numpy()

CLASS_NAMES = ["BUY", "SELL", "HOLD"]

# 1. Classification report
print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# 2. Confusion matrix & Confidence distribution
cm  = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion Matrix — Test Set")

# Confidence Distribution
for i, name in enumerate(CLASS_NAMES):
    mask = all_preds == i
    if mask.sum() > 0:
        axes[1].hist(all_confs[mask], bins=30, alpha=0.6, label=name)
axes[1].set_title("Model Confidence Distribution by Signal")
axes[1].set_xlabel("Confidence percentage")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Backtesting
Final phase is simulated real-world trading using `run_backtest(model, dataset)`.

In [ ]:
# ── Perform Trading Simulation ────────────────────────────────────────────────
from wavetrader.backtest import BacktestConfig, run_backtest

# Define risk/reward ratios & pip spread variables
bt_config = BacktestConfig(
    initial_balance=10000.0,
    risk_per_trade=0.02, # Risk 2% of portfolio per trade
    stop_loss_pips=20.0,
    take_profit_pips=40.0,
    spread_pips=1.0,
)

print(f"Executing step-by-step Backtesting Engine (Balance: ${bt_config.initial_balance})...")
try:
    results = run_backtest(model, test_dataset, bt_config, device=device)

    print("\n" + "="*40)
    print("BACKTEST RESULTS (Holdout Baseline)")
    print("="*40)
    for k, v in results.items():
        if isinstance(v, float):
            print(f"{k.ljust(20)}: {v:.4f}")
        else:
            print(f"{k.ljust(20)}: {v}")
    print("="*40)

except Exception as bt_e:
    print(f"Backtesting Engine Exception: {bt_e}")

In [ ]:
# ── Plot Equity Curve & Sample Predictions ───────────────────────────────────

if 'results' in locals():
    # 1. Equity Curve
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Try fetching the sequence from the results mapping returned from `run_backtest`
    # Typically results might be an object, dict or named tuple. Wrapping with getattr/dict logic.
    if hasattr(results, 'equity_curve'):
        equity_curve = results.equity_curve
    elif isinstance(results, dict) and 'equity_curve' in results:
        equity_curve = results['equity_curve']
    else:
        equity_curve = None
        
    if equity_curve is not None:
        ax.plot(equity_curve, lw=1.2, color="#4CAF50")
        ax.axhline(bt_config.initial_balance, linestyle="--", color="gray", alpha=0.7, label="Starting Capital")
        
        # Color zones based on profitability
        ax.fill_between(range(len(equity_curve)), bt_config.initial_balance, list(equity_curve),
                        where=[e > bt_config.initial_balance for e in equity_curve],
                        color="#4CAF50", alpha=0.15)
        ax.fill_between(range(len(equity_curve)), bt_config.initial_balance, list(equity_curve),
                        where=[e < bt_config.initial_balance for e in equity_curve],
                        color="#F44336", alpha=0.15)
                        
        ax.set_title(f"Equity Curve — Holdout Backtest")
        ax.set_xlabel("Bars Passed")
        ax.set_ylabel("Balance (USD)")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("Note: 'equity_curve' list not found in backtest results to plot.")

    # 2. Sample Predictions Logging
    print("\nVisual Logging: First 15 Sample Predictions vs Ground Truth:")
    print(f"  {'True':>6}  {'Pred':>6}  {'Conf':>6}  {'Match':>6}")
    print(f"  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}")
    for i in range(min(15, len(all_labels))):
        t, p = CLASS_NAMES[all_labels[i]], CLASS_NAMES[all_preds[i]]
        c = all_confs[i]
        match = "✓" if all_labels[i] == all_preds[i] else "✗"
        print(f"  {t:>6}  {p:>6}  {c:>6.3f}  {match:>6}")